# CALT入門

CALTは問題生成とTransformerモデルの訓練を柔軟かつ簡便に行うフレームワークです．

- **データ生成** データ生成関数を定義し，`DatasetPipeline`に渡す
- **データ読み込み** データのパスと語彙を`IOPipeline`に渡す
- **入出力の変換** `format_problem`を定義し，load preprocessorとして`IOPipeline`に渡す
- **訓練** `ModelPipeline`でモデルの準備，`TrainerPipeline`で訓練パイプラインの準備を行う

各Pipelineの設定はYAMLファイルに記述する．

> 課題：足し算をTransformerに学習させよう

整数の足し算の学習の例を通じて，CALTの基本的な使用方法を理解する．

- [src/addition.py](../src/addition.py)：問題生成関数と表記を変換する関数
- [configs/addition/](../configs/addition/)：データ生成・token化・学習の設定
- [generate.py](../src/generate.py)：データ生成
- [train.py](../src/train.py)：モデルの学習・評価

## 準備

このチュートリアルでは，Google Colabの利用を前提します

Google Driveのマウント

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

ROOT = '/content/drive/MyDrive/project-research/'  # 自身のGoogle Driveのパスを設定

%cd $ROOT

CALTのインストール

In [ ]:
%%capture
%pip install calt-x

`wandb`へのログイン（事前に[アカウント作成](https://forge.coreweave.com/wandb) & APIキーをコピー）

In [ ]:
import wandb
wandb.login()

## 1. データセット生成

Transformerモデルの学習・評価用に`（問題，正解）`のペアを大量に準備する必要がある．

1. `（問題，正解）`ペアを生成する関数を定義する
2. その関数を`DatasetPipeline`に与え，大量生成する

> 演習：最大5桁の2つの整数と，その和を与える関数`sum_problem()`を定義せよ

In [ ]:
import random

def sum_problem(seed):
    '''
    最大5桁の2つの整数とその和を返す
    
    seed: 乱数生成器のシード
    '''
    
    pass # ここにコードを書く
    
    
    return [a, b], a + b

実際にサンプルを生成してみる

In [ ]:
sum_problem(0)

この関数を`DatasetPipeline`に渡し，並列生成を実行できる

In [ ]:
from omegaconf import OmegaConf
from calt.dataset import DatasetPipeline

data_cfg = OmegaConf.load('configs/addition/data.yaml')  # データ生成の設定（サンプル数等）が書いてある
data_pipeline = DatasetPipeline.from_config(
    data_cfg,
    instance_generator=sum_problem,
)
data_pipeline.run()

configファイルの中身はこんな感じになっている

In [ ]:
print(OmegaConf.to_yaml(data_cfg))

生成したファイルを開き、内容を表示する。

In [ ]:
import json

with open(f'{data_cfg.save_dir}/train_data.jsonl') as f:
    example = json.loads(f.readline())
example

> 演習

1. [src/addition.py](../src/addition.py)に，今回の`sum_problem`を定義せよ．
2. [generate.py](../src/generate.py)の，`DatasetPipeline`に関わる記述を確認せよ．
3. [configs/addition/data.yaml](../configs/addition/data.yaml)に，どのようなパラメータがあるか確認せよ．

## 2. データの読み込みとtokenizerの準備

生成した問題と正解を読み込み，文字列をtoken IDに変換するtokenizerを用意する．

1. データの保存先と，語彙・token化の設定ファイルを指定する
2. その設定を`IOPipeline`に渡す

In [ ]:
cfg = OmegaConf.load('configs/addition/train.yaml')
print(OmegaConf.to_yaml(cfg.data))

`train_dataset_jsonl`と`test_dataset_jsonl`は，先ほど生成したファイルのパスである．
`lexer_config`で指定する[lexer.yaml](../configs/addition/lexer.yaml)には，使用する語彙と数値の分割方法を記述する．

In [ ]:
lexer_cfg = OmegaConf.load(cfg.data.lexer_config)
print(OmegaConf.to_yaml(lexer_cfg))

問題のリスト`[a, b]`と正解の整数を，学習に使う文字列に変換する関数`format_problem`を定義する．
`UserCallableLoadPreprocessor`に渡すと，データの読み込み時にこの関数が呼ばれ，その後lexerでtokenに分割される．

In [ ]:
from calt.io import IOPipeline, UserCallableLoadPreprocessor

def format_problem(problem, answer):
    a, b = problem
    return f"{a}+{b}=", str(answer)

format_problem([12, 345], 357)

設定と変換関数を`IOPipeline`に渡し，読み込んだデータを表示する．

In [ ]:
pipeline = IOPipeline.from_config(cfg.data)
pipeline.dataset_load_preprocessor = UserCallableLoadPreprocessor(format_problem)
io = pipeline.build()
io['train_dataset'][0]

読み込んだ問題をtoken IDに変換し，IDとtokenの対応を表示する．

In [ ]:
tokenizer = io['tokenizer']
problem = io['train_dataset'][0]['input']
ids = tokenizer.encode(problem, add_special_tokens=False)
list(zip(ids, tokenizer.convert_ids_to_tokens(ids)))

> 演習
> 
> 1. [configs/addition/lexer.yaml](../configs/addition/lexer.yaml)の`vocab`と`number_policy`の設定を確認せよ．
> 2. `io['test_dataset'][0]`を表示せよ．
> 3. [src/addition.py](../src/addition.py)に，今回の`format_problem`を定義せよ．
> 4. [train.py](../src/train.py)の，変換関数を`IOPipeline`に渡す記述を確認せよ．
> 

## 3. Collator：モデルに渡す直前の処理

Collatorは，データをモデルに渡す直前に，バッチごとに呼ばれる．
上の`format_problem`がデータのロード時に各サンプルに対して一度だけ実行されるのと異なり，同じデータを学習に使うたびに処理できる．例えば，毎回ランダムに入出力の表記を変えるなど，柔軟な変更ができる．

独自のCollatorを指定しない場合は，CALTが用意する標準Collatorを使う．標準Collatorは，複数の問題をまとめてtoken IDへ変換し，paddingで長さを揃え，学習用ラベルを含むバッチを作る．
ここでは標準Collatorの処理を確認する．

In [ ]:
collator = io['data_collator']

2件の問題を渡して，バッチの中身を表示する．

In [ ]:
examples = [io['train_dataset'][0], io['train_dataset'][1]]
batch = collator(examples)
batch

`input_ids`は入力のtoken ID，`attention_mask`はpaddingを区別する情報，`labels`は学習用の正解である．
`labels`の`-100`は損失計算から除外する場所を表す．

> 演習

1. バッチの`input_ids`を`tokenizer.decode()`で文字列に戻し，元の問題と比較せよ．
2. バッチの`attention_mask`と`labels`を確認せよ．

## 4. モデル生成

モデルの層数や埋め込みの次元などを[train.yaml](../configs/addition/train.yaml)の`model`に記述する．
設定と，tokenizerを含む`io`を`ModelPipeline`に渡す．

In [ ]:
print(OmegaConf.to_yaml(cfg.model))

In [ ]:
from calt.models import ModelPipeline

model = ModelPipeline.from_io_dict(cfg.model, io).build()
model

> 演習

1. 設定ファイルの層数と埋め込みの次元が，表示されたモデルのどこに対応するか確認せよ．
2. [train.py](../src/train.py)の，`ModelPipeline`に関わる記述を確認せよ．

## 5. 学習・保存

学習回数，batch size，学習率を[train.yaml](../configs/addition/train.yaml)の`train`に記述する．
`train.wandb`にはW&Bへの記録に使う名前を指定する．

| 設定 | 意味 |
|---|---|
| `project` | 記録先のプロジェクト名 |
| `group` | 関連する実行をまとめるグループ名 |
| `name` | 今回の実行名（run name） |

> 演習

[configs/addition/train.yaml](../configs/addition/train.yaml)を開き，`train.wandb`の`project`・`group`・`name`を確認せよ．変更する場合はファイルを保存し，`cfg = OmegaConf.load(...)`のセルから再実行せよ．

In [ ]:
print(OmegaConf.to_yaml(cfg.train))

### 評価指標の定義

`compute_metrics`は，評価用データへの予測と正解を受け取り，指標名と値の辞書を返す関数である．完全一致率とトークン正解率を定義する．

- 完全一致率：正解系列のすべてのtokenを当てた問題の割合．
- トークン正解率：評価対象の全tokenのうち，当てたtokenの割合．

ここでは正解の途中までを与えて次のtokenを予測する（教師強制）．答えを最初から生成する評価は「6. 評価」で行う．
`labels`が`-100`の位置を除き，終了tokenも含めて比較する．

In [ ]:
def compute_metrics(eval_prediction):
    """教師強制での完全一致率とトークン正解率を計算する。"""
    predictions = eval_prediction.predictions
    labels = eval_prediction.label_ids

    # CALTのdecoder-onlyモデルは、予測したtoken IDを返す。
    mask = labels != -100  # paddingなど、評価しない位置を除く
    correct = predictions == labels

    return {
        'exact_match': float((correct | ~mask).all(axis=1).mean()),
        'token_accuracy': float(correct[mask].mean()),
    }

> 演習

1. [src/addition_metrics.py](../src/addition_metrics.py)に，今回の`compute_metrics`を定義せよ．
2. [configs/addition/train.yaml](../configs/addition/train.yaml)の`train.eval_strategy`が`epoch`であることを確認せよ．各epoch末に評価用データで評価する設定である．

`TrainerPipeline`の`build()`後に，内部のTrainerへ`compute_metrics`を設定する．返した指標はTrainerがW&Bへ記録する．

学習設定，モデル，データとcollatorを`TrainerPipeline`に渡し，学習を開始する．

In [ ]:
from calt.trainer import TrainerPipeline

trainer = TrainerPipeline.from_io_dict(cfg.train, model, io).build()
trainer.trainer.compute_metrics = compute_metrics
trainer.train()

> 演習

1. 学習時に表示されたW&Bのリンクを開け．
2. 実行のプロジェクト名・グループ名・実行名が，`train.yaml`の`project`・`group`・`name`と一致しているか確認せよ．
3. 今回の実行に記録されたlossと，`eval/exact_match`・`eval/token_accuracy`のグラフを確認せよ．
4. [train.py](../src/train.py)の，`TrainerPipeline`に関わる記述を確認せよ．

`save_model()`でモデルとtokenizerを保存する．

In [ ]:
trainer.save_model(f'{cfg.train.output_dir}/model')

## 6. 評価

`evaluate_and_save_generation()`で評価用の問題に対する答えを生成し，正解率を求める．
生成結果は学習の出力先の`eval_results/`に保存される．
`max_length`は答えを生成する際の長さの上限で，ここでは8 tokenとする．

In [ ]:
accuracy = trainer.evaluate_and_save_generation(max_length=8)
accuracy

正解率をW&Bに記録し，今回の実行の記録を終了する．

In [ ]:
wandb.log({'accuracy': accuracy})
wandb.finish()

> 演習

1. 保存された生成結果を開き，問題に対するモデルの答えと正解を確認せよ．
2. W&Bで今回の実行を開き，`accuracy`に記録された正解率がNotebookの出力と一致しているか確認せよ．

## Pythonファイルから実行する

データ生成は[generate.py](../src/generate.py)，学習・評価は[train.py](../src/train.py)にまとめてある．
問題生成と表記の変換は[src/addition.py](../src/addition.py)から読み込む．

それぞれに，使用する設定ファイルを`--config_path`で渡す．

```bash
python -m src.generate --config_path configs/addition/data.yaml
python -m src.train --config_path configs/addition/train.yaml
```

生成したデータを使って学習するため，`generate.py`，`train.py`の順に実行する．
実験条件を切り替える場合は，[scripts/addition.sh](../scripts/addition.sh)で渡すパスを変更する．

> 演習

1. Notebookの各処理と`generate.py`・`train.py`を対応させて読め．
2. [scripts/addition.sh](../scripts/addition.sh)の実行コマンドを確認せよ．

Pythonファイルを編集して学習する場合は，Colabで次のセルを実行する．

In [ ]:
!bash scripts/addition.sh